In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
plt.style.use("fivethirtyeight")

In [ ]:
from google.colab import files
uploaded = files.upload()
print(uploaded)

In [ ]:
df = pd.read_csv("Rainfall.csv")

In [ ]:
df.head()

In [ ]:
df['rainfall'].unique()

In [ ]:
df.shape

In [ ]:
df['day'].unique()

In [ ]:
df.describe()

In [ ]:
df.describe().T

In [ ]:
df.info()

In [ ]:
df.columns=df.columns.str.strip()

In [ ]:
df.columns

In [ ]:
df = df.drop(columns = ['day'])

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df['winddirection'].unique()

In [ ]:
df['windspeed'].unique()

In [ ]:
df['winddirection'].mode()[0]

In [ ]:
df['winddirection']=df['winddirection'].fillna(df['winddirection'].mode()[0]
)

In [ ]:
df['windspeed'].median()

In [ ]:
df['windspeed'] = df['windspeed'].fillna(df['windspeed'].median())

In [ ]:
df.isnull().sum()

In [ ]:
df.columns

In [ ]:
df['rainfall'].unique()
df['rainfall']= df['rainfall'].map({"yes":1,"no":0})

In [ ]:
df.info()

In [ ]:
#EDA exploratory data analysis
df.columns

In [ ]:
columns = ['pressure', 'dewpoint', 'humidity',
       'cloud', 'sunshine', 'winddirection', 'windspeed']
plt.figure(figsize = (15,10))
for i, column in enumerate(columns,1):
       plt.subplot(3,4,i)
       sns.histplot(df[column],kde = True)
       plt.title(f"Distribution of {column}")
       plt.tight_layout()
       plt.show()

In [ ]:
plt.figure(figsize = (6,4))
sns.countplot(x = "rainfall",data = df)
plt.title("Distribution of Rainfall")
plt.show()


In [ ]:
#HANDLING MISSING VALUE -> MODE,MEAN
#CATEGORICAL -> NUMERICAL -> RAINFALL USING MAP METHOD
#KDE -> HISTPLOT
#IMBALANCE -> SMOTE (OVERSAMPLE,UNDERSAMPLE)
#FEATURE ENG-> CORRELATION MATRIX
#check any outlier is there

In [ ]:
plt.figure(figsize = (10,8))
sns.heatmap(df.corr(), annot=True, cmap = "coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
columns = ['pressure', 'dewpoint', 'humidity',
       'cloud', 'sunshine', 'winddirection', 'windspeed']
plt.figure(figsize = (15,10))
for i, column in enumerate(columns,1):
       plt.subplot(3,4,i)
       sns.boxplot(df[column])
       plt.title(f"Distribution of {column}")
       plt.tight_layout()
       plt.show()

In [ ]:
df = df.drop(columns=['maxtemp','temparature','mintemp'])

In [ ]:
df.head()

In [ ]:
df['rainfall'].value_counts()

In [ ]:
#data is imbalance both should be almost be same 1 or yes , and 0 or no


In [ ]:
df_majority = df[df['rainfall'] == 1]
df_minority = df[df['rainfall'] == 0]

In [ ]:
downsample_indices = np.random.choice(df_majority.index, size = len(df_minority),replace = False)

In [ ]:
df_majority_downsampled = df_majority.loc[downsample_indices]

In [ ]:
df_majority_downsampled.shape

In [ ]:
df_downsampled = pd.concat([df_majority_downsampled , df_minority])

In [ ]:
df_downsampled = df_downsampled.sample(frac = 1, random_state=42).reset_index(drop=True)

In [ ]:
df_downsampled['rainfall'].value_counts()

In [ ]:
#model building by using randomforest classifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score
from sklearn.ensemble import RandomForestClassifier

In [ ]:
X = df_downsampled.drop(columns = ['rainfall'])
y = df_downsampled['rainfall']

In [ ]:
X_train, X_test, y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

In [ ]:
rf_model = RandomForestClassifier(random_state=42)
param_grid_rf = {
    "n_estimators" : [50,100,200],
    "max_features" : ['sqrt','log2'],
    "max_depth" : [None, 10, 20, 30],
    "min_samples_split" : [2,5,10],
    "min_samples_leaf" : [1,2,4]
}

In [ ]:
grid_search_rf = GridSearchCV(estimator=rf_model, param_grid=param_grid_rf, cv=5, n_jobs=-1, verbose=2)

In [ ]:
grid_search_rf.fit(X_train, y_train)

In [ ]:
best_rf_model = grid_search_rf.best_estimator_

In [ ]:
best_rf_model

In [ ]:
grid_search_rf.best_params_

In [ ]:
cv_scores = cross_val_score(best_rf_model, X_train, y_train, cv=5)

In [ ]:
cv_scores

In [ ]:
np.mean(cv_scores)

In [ ]:
y_pred = best_rf_model.predict(X_test)

In [ ]:
accuracy_score(y_test,y_pred)

In [ ]:
confusion_matrix(y_test,y_pred)

In [ ]:
report_dict = classification_report(y_test,y_pred,output_dict = True)

In [ ]:
report_dict

In [ ]:
#why ml flow is used .
#consider vinay -> DS -> working on credit card fraud detection problem -> notebook 1, notebook 2, notebook 3
# kumar -> DS -> working on credit card fraud detection problem -> notebook 4, notebook 5, notebook 6
# by using mlflow we can pass our all notebook or all model with accuracy in same ui and we can compare each model or notebook

In [ ]:
pip install mlflow

In [ ]:
!pip install mlflow --quiet
!pip install pyngrok --quiet

In [ ]:
import mlflow
import mlflow.sklearn

In [ ]:
mlflow.set_experiment("Rainfall_Prediction")

In [ ]:
from google.colab import userdata
ngRoktoken = userdata.get('ngroktoken')

In [ ]:
get_ipython().system_raw ("mlflow ui --port 2000 &")
mlflow.set_tracking_uri("http://localhost:2000")
from pyngrok import ngrok
ngrok.set_auth_token(ngRoktoken)


In [ ]:
public_url = ngrok.connect(2000).public_url
print('mlflow UI URL',public_url)

In [ ]:
mlflow.set_experiment("Rainfall_Prediction")
mlflow.set_tracking_uri(uri = "http://localhost:2000")

with mlflow.start_run():
  mlflow.log_params(grid_search_rf.best_params_)
  mlflow.log_metrics({
      "accuracy":report_dict['accuracy'],
      "recall_class_0":report_dict['0']['recall'],
      "recall_class_1":report_dict['1']['recall'],
      "f1_score_macro":report_dict['macro avg']['f1-score']
  })
  mlflow.sklearn.log_model(best_rf_model,"RandomForestClassifier")

In [ ]:
model_name = "RandomForestClassifier"
run_id = "4c7c0b8f4f2b4f79862677bce93ffa53"
model_uri = f'runs:/{run_id}/{model_name}'
with mlflow.start_run(run_id = run_id):
  mlflow.register_model(model_uri = model_uri,name = model_name)

In [ ]:
import mlflow
# The previous error was due to an incorrect run ID.
# We will load the model from the registered model instead.
model_name = "RandomForestClassifier"
model_version = 1 # Or the latest version if you have multiple versions
loaded_model = mlflow.sklearn.load_model(f'models:/{model_name}/{model_version}')
loaded_model.predict(X_test)

In [ ]:
model_name = "RandomForestClassifier"
#run_id = "4c7c0b8f4f2b4f79862677bce93ffa53" # This line is no longer needed
model_uri = f'models:/{model_name}/1' # Use the models:/ URI with the version
production_model_name = 'rainfall-prediction-production'
client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri= model_uri, dst_name=production_model_name)

In [ ]:
loaded_model

In [ ]:
model_version = 1
prod_model_url = f'models:/{production_model_name}/{model_version}'
loaded_model = mlflow.sklearn.load_model(prod_model_url)
y_pred = loaded_model.predict(X_test)

In [ ]:
y_pred

In [ ]:
#predictive System
input_df = (1015.9,19.9,95,81,0.0,40.0,13.7)
input_df = pd.DataFrame([input_df],[	'pressure',	'dewpoint',	'humidity',	'cloud','sunshine',	'winddirection',	'windspeed'])

In [ ]:
prediction = loaded_model.predict(input_df)

In [ ]:
prediction

In [ ]:
print("Prediction Result: ","Rainfall" if prediction[0] == 1 else "No Rainfall")